In [8]:
import json
from pathlib import Path
from dotenv import load_dotenv
import math

import os

load_dotenv()


filming_style = "confluence_46_annotations.coco" 


annotation_folder = os.path.join(os.environ.get("PROJECT_DROPBOX"), "annotations")

json_path = os.path.join(annotation_folder, f"{filming_style}.json")
output_path = os.path.join(annotation_folder, f"{filming_style}_fixed.json")


#json_path.with_name(json_path.stem + "_fixed.json")

def to_number(x):
    """
    Convert numeric strings to int/float.
    Leave real numbers unchanged.
    Leave non-numeric values unchanged.
    """
    if isinstance(x, (int, float)):
        return x

    if isinstance(x, str):
        s = x.strip()

        # Try int first if it looks like an integer
        try:
            if s.isdigit() or (s.startswith("-") and s[1:].isdigit()):
                #print("inty")
                return int(s)
        except Exception:
            pass

        # Then try float
        try:
            #return np.floor(float(s))
            #print("floaty")
            return math.floor(float(s))
        except Exception:
            return x

    return x


with open(json_path, "r") as f:
    coco = json.load(f)

# Fix annotation bboxes and areas
for ann in coco.get("annotations", []):
    if "bbox" in ann and isinstance(ann["bbox"], list):
        ann["bbox"] = [to_number(v) for v in ann["bbox"]]

    if "area" in ann:
        ann["area"] = to_number(ann["area"])

# Fix image width/height too, just in case
for img in coco.get("images", []):
    if "width" in img:
        img["width"] = to_number(img["width"])
    if "height" in img:
        img["height"] = to_number(img["height"])

with open(output_path, "w") as f:
    json.dump(coco, f, indent=2)

print(f"Saved fixed JSON to: {output_path}")

Saved fixed JSON to: /home/berdahl/Dropbox/bear-hunting/annotations/confluence_46_annotations.coco_fixed.json
